# Microsoft Foundry: registro local de conversaciones

Notebook mínimo y ordenado para localizar una conversación guardada en `Database/conversations.json`. Está preparado para ejecutarse desde cero con **Restart → Run All**.

> Este Notebook no crea conversaciones ni envía mensajes al agente, por lo que ejecutarlo no genera una nueva respuesta del modelo.

## 1. Comprobar el kernel e inicializar el cliente

In [1]:
import json
import sys
from datetime import datetime
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = (
    "https://ricardo286malaga-9265-resource.services.ai.azure.com/"
    "api/projects/ricardo286malaga-9265"
)
AGENT_NAME = "FirstFoundry"
AGENT_VERSION = "3"

print("Python interpreter:", sys.executable)

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
openai_client = project_client.get_openai_client()

print("Microsoft Foundry clients created correctly.")
print("No request has been sent to the agent.")

Python interpreter: c:\Users\Ricardo\Documents\GitHub\AI_Agent_Microsoft01\.venv\Scripts\python.exe
Microsoft Foundry clients created correctly.
No request has been sent to the agent.


## 2. Localizar el proyecto y el registro JSON

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the nearest parent directory that contains .venv."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / ".venv").is_dir():
            return candidate

    raise FileNotFoundError(
        "Project root not found: no parent directory contains .venv."
    )


PROJECT_ROOT = find_project_root()
DATABASE_DIR = PROJECT_ROOT / "Database"
REGISTRY_PATH = DATABASE_DIR / "conversations.json"

print("Project root:", PROJECT_ROOT)
print("Registry path:", REGISTRY_PATH)

Project root: C:\Users\Ricardo\Documents\GitHub\AI_Agent_Microsoft01
Registry path: C:\Users\Ricardo\Documents\GitHub\AI_Agent_Microsoft01\Database\conversations.json


## 3. Funciones del registro

Todas las funciones se definen antes de utilizarlas, evitando depender del orden manual de ejecución de las celdas.

In [3]:
def current_timestamp() -> str:
    """Return a timezone-aware ISO 8601 timestamp."""
    return datetime.now().astimezone().isoformat(timespec="seconds")


def load_registry() -> dict:
    """Load the local registry, creating an empty structure if absent."""
    if not REGISTRY_PATH.exists():
        return {"conversations": {}}

    with REGISTRY_PATH.open("r", encoding="utf-8") as file:
        registry = json.load(file)

    if not isinstance(registry.get("conversations"), dict):
        raise ValueError("Invalid registry: 'conversations' must be an object.")

    return registry


def save_registry(registry: dict) -> None:
    """Save the registry using UTF-8 and readable indentation."""
    DATABASE_DIR.mkdir(parents=True, exist_ok=True)

    with REGISTRY_PATH.open("w", encoding="utf-8") as file:
        json.dump(registry, file, ensure_ascii=False, indent=4)
        file.write("\n")


def register_conversation(
    conversation_id: str,
    title: str,
    last_response_id: str | None,
    agent_name: str = AGENT_NAME,
    agent_version: str = AGENT_VERSION,
) -> None:
    """Create or update a conversation in the local registry."""
    registry = load_registry()
    conversations = registry["conversations"]
    timestamp = current_timestamp()
    existing = conversations.get(conversation_id, {})

    conversations[conversation_id] = {
        "title": title.strip(),
        "agent_name": agent_name,
        "agent_version": agent_version,
        "created_at": existing.get("created_at", timestamp),
        "updated_at": timestamp,
        "last_response_id": last_response_id,
    }

    save_registry(registry)


def find_conversation_by_title(title: str) -> tuple[str, dict] | None:
    """Find one registered conversation by its exact title."""
    normalized_title = title.strip().casefold()

    for conversation_id, metadata in load_registry()["conversations"].items():
        if metadata.get("title", "").strip().casefold() == normalized_title:
            return conversation_id, metadata

    return None


def get_conversation(conversation_id: str) -> dict:
    """Return metadata for one locally registered conversation."""
    conversations = load_registry()["conversations"]

    if conversation_id not in conversations:
        raise KeyError(f"Conversation {conversation_id!r} is not registered.")

    return conversations[conversation_id]


def list_registered_conversations() -> list[dict]:
    """Return registered conversations ordered by last update."""
    rows = [
        {"conversation_id": conversation_id, **metadata}
        for conversation_id, metadata
        in load_registry()["conversations"].items()
    ]

    return sorted(
        rows,
        key=lambda row: row.get("updated_at", ""),
        reverse=True,
    )

## 4. Mostrar las conversaciones registradas

In [4]:
conversations = list_registered_conversations()

print(f"Registered conversations: {len(conversations)}")

for conversation_data in conversations:
    print("=" * 70)
    print("Title:", conversation_data["title"])
    print("Conversation ID:", conversation_data["conversation_id"])
    print("Agent:", conversation_data["agent_name"])
    print("Version:", conversation_data["agent_version"])
    print("Updated at:", conversation_data["updated_at"])
    print("Last response:", conversation_data["last_response_id"])

Registered conversations: 1
Title: Payroll analysis: 2025 versus 2026
Conversation ID: conv_4c8a2a397be3cdaf00lOvPAi2pvWeGnyOG54qMzIor4NUCHYhf
Agent: FirstFoundry
Version: 3
Updated at: 2026-08-10T23:56:19+02:00
Last response: resp_4c8a2a397be3cdaf006a7a48ed94c48190bd5fdf7964df9570


## 5. Seleccionar una conversación por título

Modifica `CONVERSATION_TITLE` si quieres recuperar otra conversación.

In [5]:
CONVERSATION_TITLE = "Payroll analysis: 2025 versus 2026"

match = find_conversation_by_title(CONVERSATION_TITLE)

if match is None:
    raise LookupError(
        f"No registered conversation has the title {CONVERSATION_TITLE!r}."
    )

selected_conversation_id, _ = match
conversation_metadata = get_conversation(selected_conversation_id)

print("Selected conversation ID:", selected_conversation_id)
print(json.dumps(conversation_metadata, ensure_ascii=False, indent=4))

Selected conversation ID: conv_4c8a2a397be3cdaf00lOvPAi2pvWeGnyOG54qMzIor4NUCHYhf
{
    "title": "Payroll analysis: 2025 versus 2026",
    "agent_name": "FirstFoundry",
    "agent_version": "3",
    "created_at": "2026-08-10T23:44:38+02:00",
    "updated_at": "2026-08-10T23:56:19+02:00",
    "last_response_id": "resp_4c8a2a397be3cdaf006a7a48ed94c48190bd5fdf7964df9570"
}
